In [1]:
from datasets import load_dataset, Dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, BitsAndBytesConfig
import pandas as pd
import numpy as np
from peft import get_peft_model, LoraConfig, TaskType
import torch
import os

c:\Users\User\anaconda3\envs\summarization\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [4]:
torch.cuda.empty_cache()

In [5]:
train = pd.read_pickle('train_clean.pkl')
#validation = pd.read_pickle('validation_clean.pkl')

In [6]:
train_frag = train.sample(n=100, random_state=42)

In [7]:
len_summary = train_frag['summary'].apply(len)
len_summary.mean()

np.float64(1239.03)

In [8]:
len_summary.std()

np.float64(350.42114503678124)

In [9]:
ds_train = Dataset.from_pandas(train_frag)

In [10]:
ds_train

Dataset({
    features: ['text', 'summary', 'doc_id'],
    num_rows: 100
})

In [11]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model_name = 'google/flan-t5-large'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto").to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

c:\Users\User\anaconda3\envs\summarization\Lib\site-packages\accelerate\utils\modeling.py:804: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)
Loading weights: 100%|██████████| 558/558 [00:04<00:00, 116.26it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [12]:
tokenizer.pad_token = tokenizer.eos_token

In [13]:
token_lengths = [
    len(tokenizer(summary)["input_ids"])
    for summary in train["summary"]
]

truncated_pct = np.mean(np.array(token_lengths) > 512)

print(f"{truncated_pct}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (810 > 512). Running this sequence through the model will result in indexing errors


0.01


In [14]:
input_tokens = [
    len(tokenizer(text)["input_ids"])
    for text in train["text"].sample(min(500, len(train)))
]

print(pd.Series(input_tokens).describe())

count    500.000000
mean     507.934000
std       68.864347
min      414.000000
25%      466.000000
50%      490.000000
75%      525.000000
max      884.000000
dtype: float64


In [15]:
summary_tokens = [
    len(tokenizer(summary)["input_ids"])
    for summary in train["summary"].sample(min(500, len(train)))
]

print(pd.Series(summary_tokens).describe())

count    500.000000
mean     253.716000
std       91.020067
min       20.000000
25%      197.000000
50%      250.500000
75%      299.000000
max      929.000000
dtype: float64


In [16]:
max_len = 1024

In [17]:
def tokenize_function(x):
    texto = [f"Summarize the following text: {texto}" for texto in x['text']]
    inputs = tokenizer(texto, max_length=max_len, truncation=True)
    labels = tokenizer(x['summary'], max_length=512, truncation=True)

    inputs['labels'] = labels['input_ids']
    return inputs

In [18]:
ds_tokenized = ds_train.map(tokenize_function, batched=True, remove_columns=ds_train.column_names, batch_size=8)

Map: 100%|██████████| 100/100 [00:00<00:00, 360.49 examples/s]


In [19]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,  # para flan-t5
    r=32,                              # rango de LoRA
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q", "v", "k", "o"]         # capas a adaptar
)

In [20]:
model = get_peft_model(model, lora_config)

In [21]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100  # ignora el padding en el loss
)

In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./flan-t5-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    logging_steps=50,
    predict_with_generate=True,       # importante para seq2seq
    fp16=True                         # si tienes GPU compatible
)

In [23]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator  # aquí
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.
c:\Users\User\anaconda3\envs\summarization\Lib\site-packages\bitsandbytes\autograd\_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
